# 02 Logistic Regression Baseline

This notebook trains the baseline Logistic Regression model for airline passenger satisfaction prediction.

The baseline model is useful because it provides:

- Strong interpretability
- A simple supervised learning benchmark
- A comparison point for tree-based models

## Imports and Preprocessing

In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import GridSearchCV
from sklearn.feature_selection import RFECV
from sklearn.ensemble import RandomForestClassifier

try:
    from xgboost import XGBClassifier
except ImportError:
    print("Please install xgboost: pip install xgboost")

RANDOM_STATE = 42



# Update these paths if your CSV files are stored elsewhere.
train_path = "../data/train.csv"
test_path = "../data/test.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# Remove unnamed index columns if present
train_df = train_df.loc[:, ~train_df.columns.str.contains("^Unnamed")]
test_df = test_df.loc[:, ~test_df.columns.str.contains("^Unnamed")]

# Impute missing arrival delay values using the training median
arrival_delay_median = train_df["Arrival Delay in Minutes"].median()
train_df["Arrival Delay in Minutes"] = train_df["Arrival Delay in Minutes"].fillna(arrival_delay_median)
test_df["Arrival Delay in Minutes"] = test_df["Arrival Delay in Minutes"].fillna(arrival_delay_median)

# Encode categorical variables
nominal_cols = ["Gender", "Type of Travel"]
ordinal_cols = ["Customer Type", "Class"]

ohe = OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False)
oe = OrdinalEncoder(
    categories=[
        ["disloyal Customer", "Loyal Customer"],
        ["Eco", "Eco Plus", "Business"]
    ]
)
le = LabelEncoder()

train_nominal = pd.DataFrame(
    ohe.fit_transform(train_df[nominal_cols]),
    columns=ohe.get_feature_names_out(nominal_cols),
    index=train_df.index
)

test_nominal = pd.DataFrame(
    ohe.transform(test_df[nominal_cols]),
    columns=ohe.get_feature_names_out(nominal_cols),
    index=test_df.index
)

train_ordinal = pd.DataFrame(
    oe.fit_transform(train_df[ordinal_cols]),
    columns=ordinal_cols,
    index=train_df.index
)

test_ordinal = pd.DataFrame(
    oe.transform(test_df[ordinal_cols]),
    columns=ordinal_cols,
    index=test_df.index
)

train_encoded = train_df.drop(columns=nominal_cols + ordinal_cols).join(train_nominal).join(train_ordinal)
test_encoded = test_df.drop(columns=nominal_cols + ordinal_cols).join(test_nominal).join(test_ordinal)

train_encoded["satisfaction"] = le.fit_transform(train_encoded["satisfaction"])
test_encoded["satisfaction"] = le.transform(test_encoded["satisfaction"])

# Separate features and target
X_train = train_encoded.drop(columns=["satisfaction", "id"])
y_train = train_encoded["satisfaction"]

X_test = test_encoded.drop(columns=["satisfaction", "id"])
y_test = test_encoded["satisfaction"]

# Scale features for models that benefit from standardization
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training data shape:", X_train.shape)
print("Test data shape:", X_test.shape)
print("Target classes:", list(le.classes_))

## Base Logistic Regression

In [ ]:
logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train_scaled, y_train)

y_pred = logreg.predict(X_test_scaled)
y_proba = logreg.predict_proba(X_test_scaled)[:, 1]

logistic_results = {
    "Model": "Logistic Regression",
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1 Score": f1_score(y_test, y_pred),
    "ROC-AUC": roc_auc_score(y_test, y_proba)
}

logistic_results

## Save Results

In [ ]:
pd.DataFrame([logistic_results]).to_csv("../results/logistic_regression_results.csv", index=False)
print("Saved ../results/logistic_regression_results.csv")